In [1]:
from db.connection import connect_to_postgres_via_duckdb
from db.tables import create_clean_account_name_macro
from duckdb.sqltypes import VARCHAR

In [2]:

duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [3]:
create_clean_account_name_macro(duck)

✓ Created clean_account_name macro


# create table easybill with contact info

In [10]:
duck.sql("""
    create or replace table easybill_contacts as 
    select 
        distinct on("Kontakt: Kundennummer") 
        "Kontakt: Kundennummer", 
        "Kontakt: Firma", 
        "Kontakt: Straße/Hausnummer", 
        "Kontakt: Postleitzahl", 
        "Kontakt: Ort", 
        "Kontakt: E-Mail"
    from read_csv('data/easybill_active_contracts.csv')
""")

In [11]:
duck.sql(
    """
    select * from easybill_contacts
    """
)

┌───────────────────────┬───────────────────────────────────────────────────────────────────┬────────────────────────────┬───────────────────────┬──────────────────────────────┬──────────────────────────────────────────────┐
│ Kontakt: Kundennummer │                          Kontakt: Firma                           │ Kontakt: Straße/Hausnummer │ Kontakt: Postleitzahl │         Kontakt: Ort         │               Kontakt: E-Mail                │
│         int64         │                              varchar                              │          varchar           │        varchar        │           varchar            │                   varchar                    │
├───────────────────────┼───────────────────────────────────────────────────────────────────┼────────────────────────────┼───────────────────────┼──────────────────────────────┼──────────────────────────────────────────────┤
│             130000898 │ BIONADE GmbH                                                      │ Nordhe

# get only basic care firms in easybill (those firms might have nothing in medisoft)

In [12]:
duck.sql(
    """
    create or replace table easybill_all_docs as
    select 
        replace("Kontakt: Kundennummer", ' ', '') as id_client_easybill,
        "Kontakt: Firma" as mother_entity_easybill,
        "Posten: Artikelnummer" as id_article_easybill,
        "Posten: Artikelbeschreibung" as article_description_easybill,
    from read_csv('data/easybill_all_archived_docs.csv',types={'Kontakt: Kundennummer': 'VARCHAR'})
    order by id_client_easybill
    """
)

In [14]:
duck.sql(
    """
    select count(*) from easybill_all_docs
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        42741 │
└──────────────┘

# matching query and detection of anomalies

In [18]:
# existing tables : 
# - easybill_contacts -> contains the list of active contracts with contact info
# - easybill_all_docs -> contains the list of all docs with the client id and the mother entity

city_name = "berlin"

duck.sql(
    f"""
    with easybill_{city_name} as (
        select
            left({city_name}_active_contracts."Kd-Nr."::varchar, 9) as kd_nr,
            * exclude(column00, column01, column02) 
        from read_csv('data/{city_name}_active_contracts.csv', skip=4, strict_mode=false) as {city_name}_active_contracts
        join easybill_contacts as ec 
            on left({city_name}_active_contracts."Kd-Nr."::varchar, 9) = ec."Kontakt: Kundennummer"
        where Unternehmen is not null
    ), clean_med_firms as (
        select 
            rec_id, name,
            clean_account_name(coalesce(split(pfad, '/')[2], split(pfad, '/')[1], name)) as clean_name,
            kuerzel, pfad, strasse, plz
        from pg.medisoft.table_firmenstruktur
        where pfad like 'BSH {city_name.capitalize()}%'
    ), mother_entities as (
        select 
            kd_nr,
            Unternehmen,
            clean_account_name(Unternehmen) as entity_name
        from easybill_{city_name}
    ), matched_firms as (
        select 
            me.kd_nr, me.Unternehmen as mother_entity, me.entity_name, m.rec_id, 
            m.name, m.kuerzel, m.clean_name, m.pfad, 
            jaro_winkler_similarity(m.clean_name, me.entity_name) as sim
        from mother_entities as me
        left join clean_med_firms as m
            on m.clean_name = me.entity_name 
            or jaro_winkler_similarity(m.clean_name, me.entity_name) > 0.9
            or m.clean_name ilike '%'||me.entity_name||'%'
        where sim > 0.6
        qualify row_number() over (partition by m.rec_id order by sim desc) = 1
        order by kd_nr
    ), {city_name}_firms_found as (
        select 
            m.kd_nr as id_easybill,
            m.Unternehmen as mother_entity_easybill,
            m.entity_name as clean_entity_name_easybill,
            me.rec_id as id_medisoft,
            coalesce(me.name, me.kuerzel) as name_medisoft,
            coalesce(split(me.pfad, '/')[2], split(me.pfad, '/')[1], me.name) as mother_entity_medisoft,
            me.clean_name as clean_entity_name_medisoft,
            me.pfad as pfad_medisoft
        from mother_entities m
        join matched_firms as me
            on me.kd_nr = m.kd_nr
    ), {city_name}_firms_consolidated as (
        select
            eb.kd_nr as id_easybill,
            id_medisoft,
            Unternehmen as eb_mother_firm,
            name_medisoft,
            mother_entity_medisoft,
            pfad_medisoft,
            concat_ws(' ', "Kontakt: Straße/Hausnummer", "Kontakt: Postleitzahl", "Kontakt: Ort") as eb_address,
            concat_ws(' ', strasse, plz, ort) as medisoft_operating_firm_address,
        from easybill_{city_name} as eb 
        left join {city_name}_firms_found 
            on {city_name}_firms_found.id_easybill = eb.kd_nr
        left join pg.medisoft.table_firmenstruktur as mf
            on mf.rec_id = {city_name}_firms_found.id_medisoft
    )
    select 
        ec."Kontakt: Kundennummer" as id_easybill, 
        bf.id_medisoft,
        case when sum(
            case 
                when ead.id_article_easybill ilike '%-OMW%' then 1
                else 0
            end
        ) > 0 then true else false end as should_have_medisoft_firm,
        ec."Kontakt: Firma" as eb_mother_firm,
        bf.name_medisoft,
        bf.eb_address,
        bf.medisoft_operating_firm_address,
    from easybill_contacts as ec
    left join easybill_all_docs as ead
        on ec."Kontakt: Kundennummer" = ead.id_client_easybill
    join {city_name}_firms_consolidated as bf
        on ec."Kontakt: Kundennummer" = bf.id_easybill
    group by all
    order by 4
    """
)
#.to_csv('data/berlin_firms_found.csv')

┌─────────────┬───────────────┬───────────────────────────┬───────────────────────────────────────────────────────────────────────┬──────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                            eb_mother_firm                             │        name_medisoft         │              eb_address              │   medisoft_operating_firm_address    │
│    int64    │    varchar    │          boolean          │                                varchar                                │           varchar            │               varchar                │               varchar                │
├─────────────┼───────────────┼───────────────────────────┼───────────────────────────────────────────────────────────────────────┼──────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┤
│   113010031 │ NULL          │ fals

# end

In [136]:
duck.sql(
    """
    with joined as (
        select
            left(berlin_csv."Kd-Nr."::varchar, 9) as kd_nr,
            clean_account_name(Unternehmen) as clean_Unternehmen, 
            clean_account_name("Kontakt: Firma") as clean_Firma,
            * exclude(column00, column01, column02) 
        from read_csv('data/Berlin_Stundenerfassung.xlsx - Berlin.csv', skip=4, strict_mode=false) as berlin_csv
        join easybill_contacts as ec on left(berlin_csv."Kd-Nr."::varchar, 9) = ec."Kontakt: Kundennummer"
        where Unternehmen is not null
    ), clean_med_firms as (
        select 
            rec_id,
            name,
            regexp_replace(
                regexp_replace(
                    lower(
                        strip_accents(
                            name
                        )
                    ), 
                    '(\\b\\s+(gmbh|mbh|ag|ev|kg|kgaa|se|llp|ek|ohg|ug|inc|ltd|corp|plc)\\b).*$',
                    ''
                ),
                '[^a-z0-9]',
                '',
                'g'
            ) as clean_name,
            kuerzel,
            clean_account_name(kuerzel) as clean_kuerzel,
            pfad,
            strasse,
            plz
        from pg.medisoft.table_firmenstruktur
        where pfad like 'BSH Berlin%'
    ), mother_entities as (
        select 
            kd_nr,
            Unternehmen,
            regexp_replace(
                regexp_replace(
                    lower(
                        strip_accents(
                            Unternehmen
                        )
                    ), 
                    '(\\b\\s+(gmbh|mbh|ag|ev|kg|kgaa|se|llp|ek|ohg|ug|inc|ltd|corp|plc)\\b).*$',
                    ''
                ),
                '[^a-z0-9]',
                '',
                'g'
            ) as entity_name
        from joined
        order by length(entity_name) 
    ), daughter_entities as (
        select 
            rec_id,
            name,
            regexp_replace(
                regexp_replace(
                    lower(
                        strip_accents(
                            replace(name, '&#38;', '')
                        )
                    ), 
                    '(\\b\\s+(gmbh|mbh|ag|ev|kg|kgaa|se|llp|ek|ohg|ug|inc|ltd|corp|plc)\\b).*$',
                    ''
                ),
                '[^a-z0-9]',
                '',
                'g'
            ) as daughter_name
        from clean_med_firms
    )
    --select *,
    --    jaro_winkler_similarity(daughter_entities.daughter_name, me.entity_name) as sim
    --from daughter_entities
    --join mother_entities as me
    --on jaro_winkler_similarity(daughter_entities.daughter_name, me.entity_name) > 0.9
    --qualify row_number() over (partition by daughter_entities.rec_id order by sim desc) = 1
    select distinct on(rec_id)
        me.kd_nr,
        me.Unternehmen as mother_entity,
        me.entity_name,
        m.rec_id,
        m.name,
        m.clean_name,
        jaro_winkler_similarity(m.clean_name, me.entity_name) as sim
    from clean_med_firms as m
    join mother_entities as me
    on
     m.clean_name = me.entity_name 
     or jaro_winkler_similarity(m.clean_name, me.entity_name) > 0.9
    
    or m.clean_name ilike '%'||me.entity_name||'%'
    --where m.pfad ilike '%aios%' 
    where sim > 0.6
    qualify row_number() over (partition by m.rec_id order by sim desc) = 1

    order by kd_nr
""").to_csv('data/berlin_firms_found.csv')

In [25]:
duck.sql(
    """
    with easybill_berlin as (
        select
            left(berlin_csv."Kd-Nr."::varchar, 9) as kd_nr,
            clean_account_name(Unternehmen) as clean_Unternehmen, 
            clean_account_name("Kontakt: Firma") as clean_Firma,
            * exclude(column00, column01, column02) 
        from read_csv('data/Berlin_Stundenerfassung.xlsx - Berlin.csv', skip=4, strict_mode=false) as berlin_csv
        join easybill_contacts as ec on left(berlin_csv."Kd-Nr."::varchar, 9) = ec."Kontakt: Kundennummer"
        where Unternehmen is not null
    )
    select
        eb.kd_nr as id_easybill,
        id_medisoft,
        Unternehmen as eb_mother_firm,
        name_medisoft,
        mother_entity_medisoft,
        pfad_medisoft,
        concat_ws(' ', "Kontakt: Straße/Hausnummer", "Kontakt: Postleitzahl", "Kontakt: Ort") as eb_address,
        concat_ws(' ', strasse, plz, ort) as medisoft_operating_firm_address,
    from easybill_berlin as eb 
    left join read_csv('data/berlin_firms_found.csv') as berlin_firms_found 
        on berlin_firms_found.id_easybill = eb.kd_nr
    left join pg.medisoft.table_firmenstruktur as mf
        on mf.rec_id = berlin_firms_found.id_medisoft
    """
)
#.to_csv('data/berlin_firms_consolidated.csv')

┌─────────────┬──────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────┬────────────────────────────┬────────────────────────────────────────┬─────────────────────────────────────────────┬────────────────────────────────────────────────┐
│ id_easybill │             id_medisoft              │                                    eb_mother_firm                                    │       name_medisoft       │   mother_entity_medisoft   │             pfad_medisoft              │                 eb_address                  │        medisoft_operating_firm_address         │
│   varchar   │               varchar                │                                       varchar                                        │          varchar          │          varchar           │                varchar                 │                   varchar                   │                    varchar                  

In [144]:
duck.sql("""
select *
from pg.medisoft.table_firmenstruktur  
where pfad like 'BSH Berlin%' and pfad ilike '%ejf%'
order by trim(name)
""").to_csv('data/berlin_ejf_firms.csv')

In [206]:
duck.sql(
    """
    with splited_pfad as (
        select split(pfad, '/') as splited, pfad
        from pg.medisoft.table_firmenstruktur
        where pfad ilike '%bsh berlin%'
        and splited[2] not ilike '%Nicht Kunden%'
    )
    select 
        splited[2] as mother_entity, 
        any_value(splited)
        , count(*)
    from splited_pfad
    group by 1
    order by 3 desc
    """
)

┌─────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────┐
│                    mother_entity                    │                                              any_value(splited)                                               │ count_star() │
│                       varchar                       │                                                   varchar[]                                                   │    int64     │
├─────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────┼──────────────┤
│  EJF                                                │ ['BSH Berlin ', ' EJF ', ' Kita Regenbogen']                                                                  │           99 │
│  LebensWelt Kindertagesstätte Amendestraße gGmbH    │ ['BSH Berlin ', ' LebensWelt 

In [65]:
duck.sql(
    """
    select * from pg.medisoft.table_firmenstruktur
    where pfad ilike '%worx%'
    """
)

┌─────────┬─────────┬─────────┬─────────┬─────────┬────────────────┬───────────────┬───────────────┬─────────────────┬───────────────┬───────────────┬───────────────┬────────────────┬───────────┬────────────┬─────────────────┬─────────┬─────────┬───────────┬────────────┬─────────────────┬─────────┬─────────┬─────────┬─────────┬──────────┬─────────┬─────────┬─────────┬──────────────────┬─────────┬─────────┬─────────┬──────────┬─────────┬───────────────────┬────────────┬─────────────────────────┬────────────┬─────────────┬─────────────┬────────────────────┐
│ rec_id  │ kuerzel │  name   │  pfad   │ passiv  │ abrechnung_art │ laptop_update │ laptop_delete │ historiepflicht │ statistik_kz1 │ statistik_kz2 │ statistik_kz3 │ sync_timestamp │ sync_hash │ sequenz_nr │ farbe_verwenden │ status  │  ebene  │ has_child │ has_child2 │ inoriskrelevant │ strasse │   plz   │   ort   │ mandant │ vater_id │  staat  │ telefon │  name2  │ betreuender_arzt │   fax   │  email  │ ebene2  │ telefon2 │  name3 

In [157]:
duck.sql(
    """
    select 
    *, 
    length(split(pfad, '/')) as l 
    from pg.medisoft.table_firmenstruktur 
    where pfad ilike '%bsh berlin%' and pfad not ilike '%ejf%' and pfad not ilike '%Tesla%'
    order by pfad desc, l desc
    """
)

┌──────────────────────────────────────┬──────────────────────────────┬───────────────────────────────────┬───────────────────────────────────────────┬─────────┬────────────────┬───────────────┬───────────────┬─────────────────┬───────────────┬───────────────┬───────────────┬────────────────┬──────────────────────────────────────────┬────────────┬─────────────────┬─────────┬─────────┬───────────┬────────────┬─────────────────┬───────────────────────┬─────────┬──────────────────┬─────────┬───────────────┬─────────┬───────────────────┬─────────────────────┬──────────────────┬─────────┬─────────────────────────┬─────────┬──────────┬─────────┬───────────────────┬────────────┬─────────────────────────┬─────────────┬──────────────────────────────┬─────────────┬────────────────────┬───────┐
│                rec_id                │           kuerzel            │               name                │                   pfad                    │ passiv  │ abrechnung_art │ laptop_update │ laptop_del

In [ ]:
duck.sql(
    """
        select 
            --distinct on(id_article_easybill)
            ec."Kontakt: Kundennummer" as id_easybill, 
            bf.id_medisoft,
            case when sum(
                case 
                    when ead.id_article_easybill ilike '%-OMW%' then 1
                    else 0
                end
            ) > 0 then true else false end as should_have_medisoft_firm,

            ec."Kontakt: Firma" as eb_mother_firm,
            bf.name_medisoft,
            bf.eb_address,
            bf.medisoft_operating_firm_address,
            --ead.id_article_easybill,
            --ead.article_description_easybill,
        from easybill_contacts as ec
        left join easybill_all_docs as ead
            on ec."Kontakt: Kundennummer" = ead.id_client_easybill
        join read_csv('data/berlin_firms_consolidated.csv') as bf
            on ec."Kontakt: Kundennummer" = bf.id_easybill
        --where id_client_easybill = '130002089'
        group by all
        order by 4
    """
)
#.to_csv('data/berlin_firms_consolidated_with_docs_analysis_2.csv')
#.show(max_rows=1000)


┌─────────────┬───────────────┬───────────────────────────┬───────────────────────────────────────────────────────────────────────┬──────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┐
│ id_easybill │  id_medisoft  │ should_have_medisoft_firm │                            eb_mother_firm                             │        name_medisoft         │              eb_address              │   medisoft_operating_firm_address    │
│    int64    │    varchar    │          boolean          │                                varchar                                │           varchar            │               varchar                │               varchar                │
├─────────────┼───────────────┼───────────────────────────┼───────────────────────────────────────────────────────────────────────┼──────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┤
│   113010031 │ NULL          │ fals

In [6]:
duck.sql(
    """
    select * from easybill_all_docs where id_client_easybill = '122010004'
    """
).show(max_rows=1000)

┌────────────────────┬─────────────────────────────────────────┬─────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ id_client_easybill │         mother_entity_easybill          │ id_article_easybill │                                                                                          article_description_easybill                                                                                          │
│      varchar       │                 varchar                 │       varchar       │                                                                                                    varchar                                                                                                     │
├────────────────────┼─────────────────────────────────────────┼─────────────────────┼──────────────────────────

In [16]:
duck.sql("select * from read_csv('data/viersen_active_contracts.csv', skip=4, strict_mode=false)")

┌──────────┬──────────┬──────────┬───────┬───────────────────────────────────────────────┬──────────────┬─────────┬─────────┬───────────────┬──────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────┬─────────────────────────────────────────────┬────────────────────┬─────────────┬──────────────────┬───────────────────┬────────────────────────────────────┬────────────────────┬──────────────┬─────────────┬──────────────┬─────────────────────┬───────────────┬──────────────┬───────────────┬────────────────────┬─────────────────┬───────────────┬─────────────────┬───────────┬─────────────────┬──────────────┬────────────┬──────────────┬───────────────────┬─────────────────┬──────────────────┬──────────┐
│ column00 │ column01 │ column02 │   #   │                  Unternehmen                  │ Kundennummer │  Arzt   │  FaSi   │ Betreuungsart │                           Ansprechpartner                            │               Betriebss